# Validação do dataset experimental (estágio 09)

Valida a estrutura do conjunto de imagens e máscaras já armazenado em `MyDrive/tcc/imagens`. Esta etapa não treina nenhum modelo: ela apenas confirma contagens, correspondência entre arquivos, formato das imagens, binarização das máscaras e gera um resumo reprodutível para uso posterior no baseline U-Net.

## Bootstrap do projeto

Baixa o `src/` diretamente do repositório oficial deste TCC e prepara o workspace.

In [ ]:
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Resolução dos caminhos no Google Drive

Monta o Drive e localiza a pasta experimental definida em `src/config.yaml`.

In [ ]:
from pathlib import Path

from src import io
from src.config import get_config

config = get_config()
storage_paths = io.resolve_storage_paths()
dataset_root = storage_paths["data_embrapa"]
print(f"Dataset experimental: {dataset_root}")


## Definição dos subconjuntos

Mapeia as pastas de imagens e máscaras de treino, validação e teste.

In [ ]:
split_dirs = {
    "train": (dataset_root / "Imagens_treino", dataset_root / "Mascaras_treino"),
    "val": (dataset_root / "Imagens_validacao", dataset_root / "Mascaras_validacao"),
    "test": (dataset_root / "Imagens_teste", dataset_root / "Mascaras_teste"),
}

for split, (images_dir, masks_dir) in split_dirs.items():
    if not images_dir.is_dir() or not masks_dir.is_dir():
        raise FileNotFoundError(
            f"Estrutura ausente em {split}: {images_dir} / {masks_dir}"
        )
    print(f"{split}: OK")


## Contagem e correspondência de nomes

Confere se cada imagem possui uma máscara com o mesmo nome e registra a quantidade de pares em cada subconjunto.

In [ ]:
def png_names(folder: Path) -> list[str]:
    return sorted(path.name for path in folder.glob("*.png"))

summary = {}
for split, (images_dir, masks_dir) in split_dirs.items():
    image_names = png_names(images_dir)
    mask_names = png_names(masks_dir)
    missing_masks = sorted(set(image_names) - set(mask_names))
    missing_images = sorted(set(mask_names) - set(image_names))
    summary[split] = {
        "images": len(image_names),
        "masks": len(mask_names),
        "missing_masks": missing_masks,
        "missing_images": missing_images,
    }
    print(split, summary[split])

    if missing_masks or missing_images:
        raise ValueError(f"Pares inconsistentes no split {split}.")


## Inspeção de formato e binarização das máscaras

Abre uma amostra de cada subconjunto. As imagens são convertidas para RGB e as máscaras para tons de cinza; valores maiores que 127 são considerados classe Café.

In [ ]:
import numpy as np
from PIL import Image

samples = {}
for split, (images_dir, masks_dir) in split_dirs.items():
    image_path = sorted(images_dir.glob("*.png"))[0]
    mask_path = masks_dir / image_path.name

    with Image.open(image_path) as img:
        original_mode = img.mode
        image = np.asarray(img.convert("RGB"))

    with Image.open(mask_path) as mask_img:
        mask_mode = mask_img.mode
        mask_gray = np.asarray(mask_img.convert("L"))

    mask_binary = (mask_gray > 127).astype(np.uint8)
    coffee_ratio = float(mask_binary.mean())

    samples[split] = (image, mask_binary, image_path.name)
    print(
        f"{split}: arquivo={image_path.name}, "
        f"imagem_mode={original_mode}, mask_mode={mask_mode}, "
        f"shape={image.shape}, coffee_ratio={coffee_ratio:.4f}"
    )


## Distribuição da classe Café

Calcula a proporção de pixels positivos em todas as máscaras para identificar desbalanceamento entre Café e Não-Café.

In [ ]:
import pandas as pd

records = []
for split, (_, masks_dir) in split_dirs.items():
    for mask_path in sorted(masks_dir.glob("*.png")):
        with Image.open(mask_path) as mask_img:
            mask = np.asarray(mask_img.convert("L"))
        binary = (mask > 127).astype(np.uint8)
        records.append({
            "split": split,
            "file": mask_path.name,
            "coffee_ratio": float(binary.mean()),
            "has_coffee": bool(binary.any()),
        })

mask_stats = pd.DataFrame(records)
display(mask_stats.groupby("split")["coffee_ratio"].agg(["count", "mean", "min", "max"]))
display(mask_stats.groupby("split")["has_coffee"].agg(["sum", "count"]))


## Visualização de amostras

Exibe uma imagem e a máscara binária correspondente para cada subconjunto.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(3, 2, figsize=(10, 14))
for row, split in enumerate(("train", "val", "test")):
    image, mask, name = samples[split]
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"{split} — {name}")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Máscara binária")
    axes[row, 1].axis("off")

figure.tight_layout()


## Persistência dos artefatos

Salva a tabela de estatísticas das máscaras e a figura de amostras na árvore canônica de artefatos do TCC.

In [ ]:
figures_dir = storage_paths["artifacts_figures"]
metrics_dir = storage_paths["artifacts_metrics"]
figures_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

figure_path = figures_dir / "dataset_experimental_samples.png"
csv_path = metrics_dir / "dataset_experimental_mask_stats.csv"

figure.savefig(figure_path, dpi=150, bbox_inches="tight")
mask_stats.to_csv(csv_path, index=False)

print(f"Figura salva em: {figure_path}")
print(f"Estatísticas salvas em: {csv_path}")


## Conclusão da validação

Se todas as células forem executadas sem erro, o dataset experimental está estruturalmente pronto para o próximo estágio: implementação e treinamento da U-Net baseline.